# adapteddlo_muj — Parameter Calibration

This notebook explains **how to choose `alpha_bar` and `beta_bar`** for a physical cable.

| Parameter | Controls | Physical meaning |
|---|---|---|
| `alpha_bar` | Bending stiffness | Equal to the bending stiffness EI [N·m²] |
| `beta_bar`  | Torsional stiffness | Equal to the torsional stiffness GJ [N·m²] |

**What you will do:**
1. Understand the exact alpha/beta ↔ EI/GJ mapping derived from the source code
2. Compute alpha_bar and beta_bar from physical cable properties (radius, material)
3. Use a gravity-droop estimate to calibrate alpha_bar without hardware
4. Use the Michell Buckling Instability (MBI) formula to identify the beta/alpha ratio
5. See calibrated values for the three real cables in this repository

**Requirements:** `numpy`, `matplotlib` (Sections 1–4 need no simulation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print('numpy      ', np.__version__)
print('matplotlib ', plt.matplotlib.__version__)

## 1. The Exact Formula (from source code)

The mapping between simulation parameters and physical quantities is derived from
`adapteddlo_muj/envs/realgrav_valid_test.py` lines 89–94:

```python
J1 = np.pi * (r_thickness/2)**4 / 2.   # polar moment of area  [m⁴]
Ix = np.pi * (r_thickness/2)**4 / 4.   # second moment of area [m⁴]
stiff_vals = [beta_bar / J1,            # → G = shear modulus   [Pa]
              alpha_bar / Ix]           # → E = Young's modulus [Pa]
```

Therefore:

$$\boxed{\alpha\_bar = E \cdot I_x = E \cdot \frac{\pi}{4}\left(\frac{d}{2}\right)^4}$$

$$\boxed{\beta\_bar = G \cdot J = G \cdot \frac{\pi}{2}\left(\frac{d}{2}\right)^4}$$

where `d = r_thickness` (the value passed to `GenKin_O_weld2`), `E` is Young's modulus [Pa],
and `G = E / (2(1+ν))` is the shear modulus.

**Consequence — the ratio beta/alpha:**

$$\frac{\beta\_bar}{\alpha\_bar} = \frac{GJ}{EI} = \frac{G}{E} \cdot \frac{J}{I} = \frac{1}{2(1+\nu)} \cdot 2 = \frac{1}{1+\nu}$$

For common materials:

| Material | ν | β/α = GJ/EI |
|---|---|---|
| Rubber / silicone  | 0.50 | 0.667 |
| Nylon / HDPE       | 0.40 | 0.714 |
| PVC                | 0.38 | 0.725 |
| Steel / aluminium  | 0.30 | 0.769 |

> **Note:** Real cables (multi-strand, braided, sheathed) deviate from this isotropic formula.
> Use the empirical MBI calibration (Section 4) for best accuracy.

## 2. Calculator: Physical Properties → alpha_bar, beta_bar

In [ ]:
# ── Edit these for your cable ──────────────────────────────────────────────
d_cable   = 0.006     # r_thickness in GenKin_O_weld2 [m]  (outer diameter)
E_cable   = 2e9       # Young's modulus                [Pa] (nylon ~2 GPa, silicone ~1-50 MPa)
nu_cable  = 0.40      # Poisson's ratio                [-]  (nylon ≈ 0.4, rubber ≈ 0.5)
rho_vol   = 1150.0    # volumetric density             [kg/m³] (nylon ≈ 1100-1200)
L_cable   = 0.50      # cable length                   [m]
# ──────────────────────────────────────────────────────────────────────────

r       = d_cable / 2                        # radius [m]
A       = np.pi * r**2                       # cross-section area [m²]
Ix      = np.pi * r**4 / 4                   # second moment of area [m⁴]
J       = np.pi * r**4 / 2                   # polar moment          [m⁴]
G       = E_cable / (2 * (1 + nu_cable))     # shear modulus         [Pa]
rho_lin = rho_vol * A                        # linear density        [kg/m]

alpha_bar = E_cable * Ix
beta_bar  = G * J
b_a       = beta_bar / alpha_bar             # ≈ 1/(1+ν) for isotropic circular cross-section

print('─── Cross-section geometry ───────────────────────────────────────')
print(f'  diameter d       = {d_cable*1e3:.1f} mm')
print(f'  area A           = {A*1e6:.4f} mm²')
print(f'  I  (2nd moment)  = {Ix:.4e} m⁴')
print(f'  J  (polar)       = {J:.4e} m⁴')
print(f'  linear density   = {rho_lin*1e3:.2f} g/m')
print()
print('─── Derived stiffnesses ──────────────────────────────────────────')
print(f'  E (Young)        = {E_cable:.3e} Pa  = {E_cable/1e9:.1f} GPa')
print(f'  G (shear)        = {G:.3e} Pa  = {G/1e9:.2f} GPa')
print(f'  EI               = {E_cable*Ix:.4e} N·m²')
print(f'  GJ               = {G*J:.4e} N·m²')
print()
print('─── DER simulation parameters ────────────────────────────────────')
print(f'  alpha_bar        = {alpha_bar:.6e}   (bending stiffness EI)')
print(f'  beta_bar         = {beta_bar:.6e}   (torsional stiffness GJ)')
print(f'  beta/alpha ratio = {b_a:.3f}   (theoretical 1/(1+ν) = {1/(1+nu_cable):.3f})')
print()
print('─── Gravity droop prediction (Euler-Bernoulli cantilever) ────────')
g         = 9.81
w         = rho_lin * g                      # distributed load [N/m]
delta_tip = w * L_cable**4 / (8 * alpha_bar)
ratio     = delta_tip / L_cable
flag = '  ✓ valid regime' if ratio < 0.1 else '  ⚠ δ/L > 10%: use full DER sim'
print(f'  Tip deflection   = {delta_tip*100:.2f} cm  (δ/L = {ratio:.3f}{flag})')

## 3. Gravity Droop Calibration (no simulation needed)

For a **horizontal cantilever** under its own weight (one end clamped, gravity along −z),
Euler-Bernoulli beam theory gives the deflection profile:

$$w(x) = \frac{\rho A g}{24 \, EI}\left(6L^2 x^2 - 4Lx^3 + x^4\right)$$

Tip deflection at x = L:

$$\delta_{tip} = \frac{\rho A g\, L^4}{8\, EI}$$

Inverting: measure `δ_tip` on the real cable → compute `alpha_bar = EI`:

$$\alpha\_bar = EI = \frac{\rho A g\, L^4}{8\,\delta_{tip}} = \frac{m \cdot g \cdot L^3}{8\,\delta_{tip}}$$

where `m = ρAL` is the total mass and `L` is the cable length.

In [ ]:
# ── Edit these with your measurement ──────────────────────────────────────
L_meas     = 0.80       # cable length       [m]
m_meas     = 0.020      # total cable mass   [kg]
delta_meas = 0.05       # measured tip drop  [m]  (positive downward)
# ──────────────────────────────────────────────────────────────────────────

g = 9.81
alpha_from_droop = m_meas * g * L_meas**3 / (8 * delta_meas)
print(f'Measured tip deflection : {delta_meas*100:.1f} cm')
print(f'Inferred alpha_bar (EI) : {alpha_from_droop:.4e} N·m²')

# ── Sensitivity plot: shape vs alpha_bar ──────────────────────────────────
x = np.linspace(0, L_meas, 300)
w_line = m_meas * g / L_meas           # linear load [N/m]

alpha_sweep = alpha_from_droop * np.array([0.25, 0.5, 1.0, 2.0, 4.0])
labels      = ['¼×', '½×', '1× (fitted)', '2×', '4×']
colors      = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(alpha_sweep)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for alpha_val, lbl, c in zip(alpha_sweep, labels, colors):
    deflection = w_line / (24 * alpha_val) * (6*L_meas**2*x**2 - 4*L_meas*x**3 + x**4)
    lw = 2.5 if '1×' in lbl else 1.5
    ax.plot(x * 100, -deflection * 100, color=c, lw=lw, label=lbl)

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('x along cable [cm]', fontsize=11)
ax.set_ylabel('Deflection [cm]  (downward +)', fontsize=10)
ax.set_title('Gravity droop shape\nvs alpha_bar scaling', fontsize=11)
ax.legend(title='alpha_bar', fontsize=9)
ax.grid(True, alpha=0.3)
ax.invert_yaxis()

# ── Panel 2: tip deflection vs alpha_bar ──────────────────────────────────
ax2 = axes[1]
alpha_range = np.logspace(
    np.log10(alpha_from_droop) - 1.5,
    np.log10(alpha_from_droop) + 1.5,
    200
)
delta_range = w_line * L_meas**4 / (8 * alpha_range)

ax2.loglog(alpha_range, delta_range * 100, 'b-', lw=2, label='Euler–Bernoulli')
ax2.axvline(alpha_from_droop, color='r', ls='--', lw=1.5,
            label=f'fitted ᾱ = {alpha_from_droop:.2e}')
ax2.axhline(delta_meas * 100, color='g', ls=':', lw=1.5,
            label=f'measured δ = {delta_meas*100:.0f} cm')
ax2.set_xlabel('alpha_bar  [N·m²]', fontsize=11)
ax2.set_ylabel('Tip deflection  [cm]', fontsize=11)
ax2.set_title('Calibration curve\n(tip deflection vs bending stiffness)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('gravity_droop_calibration.pdf', bbox_inches='tight')
plt.show()
print(f'Figure saved to gravity_droop_calibration.pdf')

## 4. Torsional Stiffness from Michell Buckling Instability (MBI)

For a **straight rod with both ends clamped** that is twisted by θ turns at one end,
Kirchhoff rod theory predicts the critical twist at which the rod spontaneously buckles:

$$\theta_{crit} = \frac{2\pi\sqrt{3}}{\beta\_bar / \alpha\_bar} = \frac{2\pi\sqrt{3}}{GJ/EI}$$

This is the formula used in `scripts/dlo_testdata.py` (`mbi_data` and `mbi_plot` functions).

**Procedure:**
1. Clamp both ends of the cable horizontally
2. Slowly rotate one end; record the angle at which the rod jumps to a helical shape
3. Use `theta_crit` to solve for `beta_bar = alpha_bar × 2π√3 / theta_crit`

In [ ]:
# ── Edit: fill in your measured alpha_bar and critical twist ───────────────
alpha_known   = alpha_from_droop   # from gravity droop above  [N·m²]
theta_crit_measured = 18.5         # measured buckling angle   [rad]
# ──────────────────────────────────────────────────────────────────────────

b_a_from_mbi  = 2 * np.pi * np.sqrt(3) / theta_crit_measured
beta_from_mbi = alpha_known * b_a_from_mbi

print(f'Critical twist (measured) : {theta_crit_measured:.2f} rad '
      f'= {theta_crit_measured/(2*np.pi):.2f} full turns')
print(f'Inferred beta/alpha ratio : {b_a_from_mbi:.3f}')
print(f'  theoretical (ν=0.5)     : {1/1.5:.3f}  (rubber)')
print(f'  theoretical (ν=0.3)     : {1/1.3:.3f}  (steel)')
print(f'Inferred beta_bar (GJ)    : {beta_from_mbi:.4e} N·m²')

# ── Visualization: theta_crit vs b/a ratio ────────────────────────────────
b_a_range = np.linspace(0.3, 3.0, 300)
theta_range = 2 * np.pi * np.sqrt(3) / b_a_range

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(b_a_range, theta_range, 'b-', lw=2, label='Kirchhoff formula')
ax.axhline(theta_crit_measured, color='r', ls='--', lw=1.5,
           label=f'measured θ_crit = {theta_crit_measured:.1f} rad')
ax.axvline(b_a_from_mbi, color='g', ls=':', lw=1.5,
           label=f'inferred β/α = {b_a_from_mbi:.3f}')

for nu_val, mat in [(0.50, 'rubber  ν=0.50'), (0.40, 'nylon   ν=0.40'),
                    (0.30, 'steel   ν=0.30')]:
    ax.axvline(1/(1+nu_val), color='gray', ls=':', lw=0.8, alpha=0.7)
    ax.text(1/(1+nu_val)+0.02, theta_range.max()*0.92, mat,
            fontsize=7, color='gray')

ax.set_xlabel('β/α = GJ / EI  (torsion-to-bending ratio)', fontsize=11)
ax.set_ylabel('Critical end-twist  θ_crit  [rad]', fontsize=11)
ax.set_title('Michell Buckling Instability — calibration curve\n'
             'θ_crit = 2π√3 / (β/α)', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0.3, 3.0)
ax.set_ylim(0, 40)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mbi_calibration.pdf', bbox_inches='tight')
plt.show()
print(f'Figure saved to mbi_calibration.pdf')

## 5. Calibrated Values — Repository Cable Database

The repository ships calibrated values for three real cables identified via
gravity-droop (Step 1) and MBI twist (Step 2). Source: `scripts/real2sim_paramiden.py`.

Cable geometry used in those experiments: `r_len = 2π ≈ 6.28 m`, `r_thickness = 0.03 m`.

In [ ]:
# From real2sim_paramiden.py (adapted controller values)
cables = {
    'white': dict(alpha=5.644018525809911e-05, beta=3.837932597550739e-05, b_a=0.68),
    'black': dict(alpha=0.0009997975674291843, beta=0.001099777324172103,  b_a=1.10),
    'red'  : dict(alpha=0.00063696780505569,   beta=0.001203869151555254,  b_a=1.89),
}

# Back-compute E, G, and tip deflection
r_calib     = 0.03 / 2           # r_thickness/2
Ix_calib    = np.pi * r_calib**4 / 4
J_calib     = np.pi * r_calib**4 / 2
L_calib     = 2 * np.pi          # 6.28 m

print(f'Cable geometry: r_thickness=0.03 m  →  I={Ix_calib:.3e} m⁴  J={J_calib:.3e} m⁴')
print()
print(f'{"Cable":<8}  {"alpha_bar":>12}  {"beta_bar":>12}  {"β/α":>6}  '
      f'{"E [kPa]":>10}  {"G [kPa]":>10}  {"ν_eff":>7}')
print('─' * 80)
for name, c in cables.items():
    E = c['alpha'] / Ix_calib
    G = c['beta']  / J_calib
    nu_eff = E / (2 * G) - 1        # from G = E/(2(1+ν)) → ν = E/(2G)-1
    print(f'{name:<8}  {c["alpha"]:12.4e}  {c["beta"]:12.4e}  '
          f'{c["b_a"]:6.2f}  {E/1e3:10.1f}  {G/1e3:10.1f}  {nu_eff:7.3f}')

### Observations from the cable database

- **White wire** — very soft (E ≈ 1.4 kPa), rubber-like ratio β/α ≈ 0.68 ≈ 1/1.47  
  → ν_eff ≈ 0.47 (nearly incompressible, consistent with silicone rubber)
- **Black wire** — stiffer by 18×, but still flexible; β/α = 1.10 > 2/3  
  → ν_eff ≈ −0.05 which is unphysical for an isotropic rod, suggesting the black cable
  has twisted strands or a braided structure that gives it extra torsional resistance
- **Red wire** — β/α = 1.89, very high torsional stiffness relative to bending;  
  common for coaxial cables or spring-wound cables

> **β/α > 1 is not achievable for a solid isotropic cylinder** (max is 1 for ν→0),
> but is physically realistic for multi-strand ropes, coaxial cables, and wound springs.
> Always use the empirical MBI calibration rather than the ν formula for real cables.

In [ ]:
# ── Visual comparison of the three calibrated cables ──────────────────────
# Simulate gravity droop shape for each cable (Euler-Bernoulli, same length)

x = np.linspace(0, L_calib, 300)
rho_A_g = 0.10  # assume 100 g/m linear density for illustration

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: gravity droop shapes ────────────────────────────────────────────
ax = axes[0]
cable_colors = {'white': 'goldenrod', 'black': 'dimgray', 'red': 'tomato'}
for name, c in cables.items():
    defl = rho_A_g / (24 * c['alpha']) * (
        6*L_calib**2*x**2 - 4*L_calib*x**3 + x**4
    )
    ax.plot(x, -defl, color=cable_colors[name], lw=2, label=f'{name}  α={c["alpha"]:.1e}')

ax.set_xlabel('Arc length  s  [m]', fontsize=11)
ax.set_ylabel('Deflection  δ  [m]  (downward +)', fontsize=10)
ax.set_title(f'Gravity droop shape (Euler–Bernoulli)\n'
             f'linear load = {rho_A_g:.2f} N/m,  L = {L_calib:.2f} m', fontsize=11)
ax.legend(fontsize=9)
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# ── Right: MBI critical twist ─────────────────────────────────────────────
ax2 = axes[1]
b_a_arr = np.array([c['b_a'] for c in cables.values()])
names   = list(cables.keys())
colors  = [cable_colors[n] for n in names]
theta_crits = 2 * np.pi * np.sqrt(3) / b_a_arr

b_a_range = np.linspace(0.3, 2.5, 300)
ax2.plot(b_a_range, 2*np.pi*np.sqrt(3)/b_a_range, 'k-', lw=1.5, label='Kirchhoff formula')
ax2.scatter(b_a_arr, theta_crits, c=colors, s=120, zorder=5)
for name, ba, tc in zip(names, b_a_arr, theta_crits):
    ax2.annotate(f' {name}\nβ/α={ba}', (ba, tc), fontsize=8)

# Isotropic bounds
for nu_val, lbl in [(0.5,'rubber\nν=0.5'), (0.3,'steel\nν=0.3'), (0.0,'ν=0')]:
    ax2.axvline(1/(1+nu_val), color='gray', ls=':', lw=0.8, alpha=0.6)
    ax2.text(1/(1+nu_val)+0.02, 34, lbl, fontsize=7, color='gray')

ax2.axvspan(0.3, 1.0, alpha=0.06, color='blue', label='isotropic range')
ax2.axvspan(1.0, 2.5, alpha=0.06, color='red',  label='structured cables')
ax2.set_xlabel('β/α = GJ/EI  (β_bar / α_bar)', fontsize=11)
ax2.set_ylabel('MBI critical twist  θ_crit  [rad]', fontsize=11)
ax2.set_title('MBI critical twist for the three repository cables', fontsize=11)
ax2.legend(fontsize=8)
ax2.set_xlim(0.3, 2.5)
ax2.set_ylim(0, 38)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cable_database_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved to cable_database_comparison.pdf')

## 6. Calibration Procedure Summary

| Step | Measurement | Formula | Gives |
|---|---|---|---|
| 1a | Cable radius `r`, material `E` | α = E·π(d/2)⁴/4 | `alpha_bar` (first estimate) |
| 1b | Gravity droop `δ_tip`, mass `m`, length `L` | α = m·g·L³/(8δ) | `alpha_bar` (measured) |
| 2a | Material Poisson ratio `ν` | β/α = 1/(1+ν) | `beta_bar` (first estimate) |
| 2b | MBI critical twist `θ_crit` | β/α = 2π√3/θ_crit | `beta_bar` (measured) |

**For full MuJoCo-based calibration** (gravity droop + MBI with the actual simulation,
including collision and damping effects): run `scripts/real2sim_paramiden.py` with
`--testtype mbi_teststiff` (bending) or `--testtype mbi` (torsion).

### Quick-start values

If you don't have access to the physical cable and just want to simulate:

```python
# Thin flexible cable (silicone/rubber, d=6mm, E=5MPa)
alpha_bar = 3.2e-6    # EI
beta_bar  = 2.1e-6    # GJ  (β/α ≈ 0.67, rubber-like)

# Medium cable (HDPE, d=10mm, E=800MPa)
alpha_bar = 3.9e-3
beta_bar  = 2.8e-3    # β/α ≈ 0.71

# LHB validation rod (from data/lhb/adapt/, d=40mm, E≈670kPa)
alpha_bar = 1.345
beta_bar  = 0.789     # β/α ≈ 0.59
```

See `notebooks/quickstart.ipynb` for how these parameters affect the LHB shape.